# NLLB-200 Fine-tuning for Legal Nepali-English Translation

**Model:** facebook/nllb-200-distilled-600M  
**Configurations:** 3 (NE→EN, EN→NE, Bidirectional)  
**Dataset:** Low-resource legal domain (~5K pairs)

## Setup
1. Runtime → **A100 GPU** + **High RAM**
2. Ensure data exists at: `/MyDrive/Legal_NLP/data/splits/`
3. Run all cells sequentially

**Models saved to:** `/content/models/` (local runtime)  
**Metrics saved to:** `/MyDrive/Legal_NLP/results/nllb/` (persistent)

---

## Cell 1: Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install packages
print("Installing packages...")
!pip install -q torch transformers datasets sentencepiece
!pip install -q pandas numpy scikit-learn

# Check GPU
import torch
print(f"\n{'='*60}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ WARNING: No GPU! Enable GPU runtime.")
print(f"{'='*60}")

# Set paths
DRIVE_BASE = '/content/drive/MyDrive/Legal_NLP'

# Create directories
import os
os.makedirs(f"{DRIVE_BASE}/results/nllb/metrics", exist_ok=True)
os.makedirs("/content/models", exist_ok=True)

print(f"\n✅ Setup complete!")
print(f"Data: {DRIVE_BASE}/data/splits/")
print(f"Results: {DRIVE_BASE}/results/nllb/")
print(f"Models: /content/models/ (local runtime only)")

## Cell 2: Training Function (NLLB-200)

In [ ]:
from transformers import (
    AutoModelForSeq2SeqLM, AutoTokenizer,
    get_constant_schedule_with_warmup
)
from transformers.optimization import Adafactor
from datasets import Dataset
import pandas as pd
import json
from datetime import datetime
import torch
import numpy as np
from tqdm import tqdm
import random

def train_nllb(direction, output_dir, train_data_path, val_data_path, resume_from_checkpoint=None):
    """
    Train NLLB-200 for legal translation using MANUAL training loop.
    (Seq2SeqTrainer has bugs with NLLB - manual loop works better)
    
    Args:
        direction: 'ne_en', 'en_ne', or 'bidirectional'
        output_dir: Local path to save checkpoints
        train_data_path: Path to train.csv
        val_data_path: Path to val.csv
        resume_from_checkpoint: Path to resume from (optional)
    """
    print("="*80)
    print(f"Training NLLB-200: {direction.upper()} (Manual Training Loop)")
    print(f"Output: {output_dir}")
    print("="*80)

    # Load model and tokenizer
    model_name = "facebook/nllb-200-distilled-600M"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    if resume_from_checkpoint and os.path.exists(resume_from_checkpoint):
        print(f"Loading from checkpoint: {resume_from_checkpoint}")
        model = AutoModelForSeq2SeqLM.from_pretrained(resume_from_checkpoint)
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    
    model = model.cuda()
    print(f"Parameters: {model.num_parameters():,}")

    # Load data
    train_df = pd.read_csv(train_data_path)
    val_df = pd.read_csv(val_data_path)

    # Language codes for NLLB
    lang_codes = {'en': 'eng_Latn', 'ne': 'npi_Deva'}

    # Optimizer (using Adafactor like the working example)
    optimizer = Adafactor(
        [p for p in model.parameters() if p.requires_grad],
        scale_parameter=False,
        relative_step=False,
        lr=3e-5,
        clip_threshold=1.0,
        weight_decay=0.01,
    )
    scheduler = get_constant_schedule_with_warmup(optimizer, num_warmup_steps=200)

    # Training parameters
    batch_size = 4
    max_length = 128
    num_epochs = 6
    gradient_accumulation_steps = 8
    logging_steps = 50
    save_steps = 200
    
    total_steps = (len(train_df) // batch_size) * num_epochs
    
    print(f"\\nTraining config:")
    print(f"  Epochs: {num_epochs}")
    print(f"  Batch size: {batch_size}")
    print(f"  Gradient accumulation: {gradient_accumulation_steps}")
    print(f"  Total steps: {total_steps}")
    print(f"  Train samples: {len(train_df)}")
    print(f"  Val samples: {len(val_df)}")

    def get_batch(batch_size, data, direction_override=None):
        """Get a batch of translation pairs"""
        dir_to_use = direction_override if direction_override else direction
        
        if dir_to_use == 'ne_en':
            src_col, tgt_col = 'Nepali', 'English'
            src_lang, tgt_lang = lang_codes['ne'], lang_codes['en']
        elif dir_to_use == 'en_ne':
            src_col, tgt_col = 'English', 'Nepali'
            src_lang, tgt_lang = lang_codes['en'], lang_codes['ne']
        else:  # bidirectional - randomly choose direction
            if random.random() < 0.5:
                src_col, tgt_col = 'Nepali', 'English'
                src_lang, tgt_lang = lang_codes['ne'], lang_codes['en']
            else:
                src_col, tgt_col = 'English', 'Nepali'
                src_lang, tgt_lang = lang_codes['en'], lang_codes['ne']
        
        indices = random.sample(range(len(data)), min(batch_size, len(data)))
        sources = [str(data.iloc[i][src_col]) for i in indices]
        targets = [str(data.iloc[i][tgt_col]) for i in indices]
        
        return sources, targets, src_lang, tgt_lang

    # Training loop
    model.train()
    losses = []
    global_step = 0
    best_val_loss = float('inf')
    
    print("\\nStarting training...")
    
    for epoch in range(num_epochs):
        print(f"\\n{'='*80}")
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print(f"{'='*80}")
        
        epoch_losses = []
        pbar = tqdm(range(len(train_df) // batch_size), desc=f"Epoch {epoch+1}")
        
        for step in pbar:
            try:
                # Get batch
                xx, yy, src_lang, tgt_lang = get_batch(batch_size, train_df)
                
                # Tokenize source
                tokenizer.src_lang = src_lang
                x = tokenizer(xx, return_tensors='pt', padding=True, 
                            truncation=True, max_length=max_length).to(model.device)
                
                # Tokenize target
                tokenizer.src_lang = tgt_lang
                y = tokenizer(yy, return_tensors='pt', padding=True,
                            truncation=True, max_length=max_length).to(model.device)
                
                # Replace padding with -100 (ignored in loss)
                labels = y.input_ids.clone()
                labels[labels == tokenizer.pad_token_id] = -100
                
                # Forward pass
                loss = model(**x, labels=labels).loss
                loss = loss / gradient_accumulation_steps
                loss.backward()
                
                losses.append(loss.item() * gradient_accumulation_steps)
                epoch_losses.append(loss.item() * gradient_accumulation_steps)
                
                # Update weights every gradient_accumulation_steps
                if (step + 1) % gradient_accumulation_steps == 0:
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    global_step += 1
                
                # Logging
                if global_step % logging_steps == 0 and global_step > 0:
                    avg_loss = np.mean(losses[-logging_steps:])
                    pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'step': global_step})
                
                # Save checkpoint
                if global_step % save_steps == 0 and global_step > 0:
                    checkpoint_path = f"{output_dir}/checkpoint-{global_step}"
                    os.makedirs(checkpoint_path, exist_ok=True)
                    model.save_pretrained(checkpoint_path)
                    tokenizer.save_pretrained(checkpoint_path)
                    print(f"\\n  Checkpoint saved: {checkpoint_path}")
                
            except RuntimeError as e:
                print(f"\\nError at step {step}: {e}")
                optimizer.zero_grad()
                torch.cuda.empty_cache()
                continue
        
        # End of epoch
        avg_epoch_loss = np.mean(epoch_losses)
        print(f"\\nEpoch {epoch + 1} completed - Avg train loss: {avg_epoch_loss:.4f}")
        
        # Validation
        print("Running validation...")
        model.eval()
        val_losses = []
        
        with torch.no_grad():
            for _ in tqdm(range(min(50, len(val_df) // batch_size)), desc="Validation"):
                try:
                    xx, yy, src_lang, tgt_lang = get_batch(batch_size, val_df)
                    
                    tokenizer.src_lang = src_lang
                    x = tokenizer(xx, return_tensors='pt', padding=True,
                                truncation=True, max_length=max_length).to(model.device)
                    
                    tokenizer.src_lang = tgt_lang
                    y = tokenizer(yy, return_tensors='pt', padding=True,
                                truncation=True, max_length=max_length).to(model.device)
                    
                    labels = y.input_ids.clone()
                    labels[labels == tokenizer.pad_token_id] = -100
                    
                    loss = model(**x, labels=labels).loss
                    val_losses.append(loss.item())
                except:
                    continue
        
        avg_val_loss = np.mean(val_losses) if val_losses else 999
        print(f"Validation loss: {avg_val_loss:.4f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_path = f"{output_dir}/best_model"
            os.makedirs(best_path, exist_ok=True)
            model.save_pretrained(best_path)
            tokenizer.save_pretrained(best_path)
            print(f"✅ Best model saved: {best_path}")
        
        model.train()
    
    # Save final model
    final_path = f"{output_dir}/final_model"
    os.makedirs(final_path, exist_ok=True)
    model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)
    
    # Save metrics
    metrics = {
        'model': 'nllb200',
        'direction': direction,
        'final_train_loss': np.mean(losses[-100:]) if losses else 0,
        'final_val_loss': best_val_loss,
        'total_steps': global_step,
        'timestamp': datetime.now().isoformat()
    }
    
    metrics_path = f"{DRIVE_BASE}/results/nllb/metrics/nllb_{direction}_training_metrics.json"
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    
    print(f"\\n{'='*80}")
    print(f"✅ Training complete!")
    print(f"✅ Final model: {final_path}")
    print(f"✅ Best model: {best_path}")
    print(f"✅ Metrics: {metrics_path}")
    print(f"{'='*80}")
    
    return model

print("✅ Training function defined (MANUAL LOOP - no Seq2SeqTrainer)")

## Cell 3: Train NE→EN

In [ ]:
output_dir = "/content/models/nllb200_ne_en"
checkpoint = None
if os.path.exists(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-')]
    if checkpoints:
        checkpoint = os.path.join(output_dir, sorted(checkpoints)[-1])
        print(f"Resuming from: {checkpoint}\n")

train_nllb('ne_en', output_dir,
           f'{DRIVE_BASE}/data/splits/train.csv',
           f'{DRIVE_BASE}/data/splits/val.csv', checkpoint)

## Cell 4: Train EN→NE

In [ ]:
output_dir = "/content/models/nllb200_en_ne"
checkpoint = None
if os.path.exists(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-')]
    if checkpoints:
        checkpoint = os.path.join(output_dir, sorted(checkpoints)[-1])
        print(f"Resuming from: {checkpoint}\n")

train_nllb('en_ne', output_dir,
           f'{DRIVE_BASE}/data/splits/train.csv',
           f'{DRIVE_BASE}/data/splits/val.csv', checkpoint)

## Cell 5: Train Bidirectional

In [ ]:
output_dir = "/content/models/nllb200_bidirectional"
checkpoint = None
if os.path.exists(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-')]
    if checkpoints:
        checkpoint = os.path.join(output_dir, sorted(checkpoints)[-1])
        print(f"Resuming from: {checkpoint}\n")

train_nllb('bidirectional', output_dir,
           f'{DRIVE_BASE}/data/splits/train.csv',
           f'{DRIVE_BASE}/data/splits/val.csv', checkpoint)

print("\n" + "="*80)
print("🎉 ALL NLLB-200 TRAINING COMPLETE!")
print("="*80)
print("\nNext steps:")
print("1. Ensure mbart50_training.ipynb is complete")
print("2. Run results_analysis.ipynb to evaluate and generate paper figures")
print("="*80)